# 04. 접수 수요 일자/지역 분석

2025년 일자별 접수 추세와 출발구/목적구 기반 지역별 요청 집중도를 분석한다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

df_request = pd.read_csv(
    'data/서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv',
    parse_dates=['접수일시', '접수일자']
)

df_request.info()
df_request.head()


In [ ]:
missing_request_datetime_count = df_request['접수일시'].isna().sum()

print(f'접수일시 결측 행 수: {missing_request_datetime_count:,}건')

In [ ]:
request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

invalid_request_datetime = df_request[
    request_dt.isna() & df_request['접수일시'].notna()
]

print(f'접수일시 날짜 형식 변환 실패 행 수: {len(invalid_request_datetime):,}건')

#### 일자별 접수건수의 z-score
1. 날짜별 접수건수를 계산
2. 전체 일평균 접수건수 계산
3. 전체 일별 접수건수의 표준편차 계산
4. 각 날짜가 평균에서 표준편차 몇 개만큼 떨어져 있는지 계산
5. z_score 절댓값이 3 이상이면 이상치 후보로 판단

전체 평균 기준 IQR만 쓰면 출근시간이 전부 이상치로 잡힐 수 있어서, 요일·시간대별 기준선을 만든 뒤 그 기준에서 벗어난 정도를 봄

z_score = (해당 날짜 접수건수 - 일평균 접수건수) / 표준편차

z_score >= 3   : 비정상적으로 접수건수가 많은 날짜 후보
z_score <= -3  : 비정상적으로 접수건수가 적은 날짜 후보
|z_score| >= 3 : 전체 이상치 후보

In [ ]:
request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

daily_request_count = (
    df_request.assign(접수일자=request_dt.dt.date)
    .dropna(subset=['접수일자'])
    .groupby('접수일자')
    .size()
    .reset_index(name='접수건수')
)

mean_count = daily_request_count['접수건수'].mean()
std_count = daily_request_count['접수건수'].std()

daily_request_count['z_score'] = (
    daily_request_count['접수건수'] - mean_count
) / std_count

outlier_dates = daily_request_count[
    daily_request_count['z_score'].abs() >= 3
].sort_values('z_score', ascending=False)

print(f'일평균 접수건수: {mean_count:,.2f}')
print(f'표준편차: {std_count:,.2f}')
print(f'이상치 후보 날짜 수: {len(outlier_dates):,}')

### 일자별 접수건수 추세 및 이상치

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

df_request_2025 = df_request[
    (request_dt >= '2025-01-01') &
    (request_dt < '2026-01-01')
].copy()

daily_request_count_2025 = (
    df_request_2025.assign(
        접수일자=pd.to_datetime(df_request_2025['접수일시'], errors='coerce').dt.date
    )
    .dropna(subset=['접수일자'])
    .groupby('접수일자')
    .size()
    .reset_index(name='접수건수')
)

daily_request_count_2025['접수일자'] = pd.to_datetime(daily_request_count_2025['접수일자'])
daily_request_count_2025['접수월'] = daily_request_count_2025['접수일자'].dt.month
daily_request_count_2025['일'] = daily_request_count_2025['접수일자'].dt.day
daily_request_count_2025['요일번호'] = daily_request_count_2025['접수일자'].dt.dayofweek
daily_request_count_2025['주말여부'] = daily_request_count_2025['요일번호'].isin([5, 6])

fig, axes = plt.subplots(3, 4, figsize=(20, 12), sharey=True)
axes = axes.flatten()

for month in range(1, 13):
    ax = axes[month - 1]

    month_data = daily_request_count_2025[
        daily_request_count_2025['접수월'] == month
    ]

    weekend_data = month_data[month_data['주말여부']]

    ax.plot(
        month_data['일'],
        month_data['접수건수'],
        marker='o',
        linewidth=1.8,
        markersize=3,
        color='#4c78a8',
        label='일자별 접수건수'
    )

    ax.scatter(
        weekend_data['일'],
        weekend_data['접수건수'],
        color='red',
        s=35,
        label='주말'
    )

    ax.set_title(f'{month}월')
    ax.set_xlabel('일')
    ax.set_ylabel('접수건수')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xlim(1, 31)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')

plt.suptitle('2025년 월별 일자별 접수건수 추세: 주말 표시', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

print(f"분석 시작일: {daily_request_count_2025['접수일자'].min().date()}")
print(f"분석 종료일: {daily_request_count_2025['접수일자'].max().date()}")
print(f"분석 일수: {daily_request_count_2025['접수일자'].nunique():,}일")
print(f"총 접수건수: {daily_request_count_2025['접수건수'].sum():,}건")


In [ ]:
request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

daily_request_count = (
    df_request.assign(접수일자=request_dt.dt.date)
    .dropna(subset=['접수일자'])
    .groupby('접수일자')
    .size()
    .reset_index(name='접수건수')
)

daily_request_count['접수일자'] = pd.to_datetime(daily_request_count['접수일자'])

max_request_day = daily_request_count.loc[
    daily_request_count['접수건수'].idxmax()
]

min_request_day = daily_request_count.loc[
    daily_request_count['접수건수'].idxmin()
]

top_request_days = daily_request_count.sort_values(
    '접수건수',
    ascending=False
).head(10)

bottom_request_days = daily_request_count.sort_values(
    '접수건수',
    ascending=True
).head(10)

print('최대 접수일')
display(max_request_day.to_frame().T)

print('최소 접수일')
display(min_request_day.to_frame().T)

print('접수건수 상위 날짜 TOP 10')
display(top_request_days)

print('접수건수 하위 날짜 TOP 10')
display(bottom_request_days)

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

daily_request_count = (
    df_request.assign(접수일자=request_dt.dt.date)
    .dropna(subset=['접수일자'])
    .groupby('접수일자')
    .size()
    .reset_index(name='접수건수')
)

daily_request_count['접수일자'] = pd.to_datetime(daily_request_count['접수일자'])

plt.figure(figsize=(16, 6))

plt.plot(
    daily_request_count['접수일자'],
    daily_request_count['접수건수'],
    color='#4c78a8',
    linewidth=1.8,
    marker='o',
    markersize=3
)

plt.title('일별 접수건수 추세')
plt.xlabel('접수일자')
plt.ylabel('접수건수')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 일자별 접수 추세

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

df_request_2025 = df_request[
    (request_dt >= '2025-01-01') &
    (request_dt < '2026-01-01')
].copy()

daily_request_count = (
    df_request_2025.assign(
        접수일자=pd.to_datetime(df_request_2025['접수일시'], errors='coerce').dt.date
    )
    .dropna(subset=['접수일자'])
    .groupby('접수일자')
    .size()
    .reset_index(name='접수건수')
)

daily_request_count['접수일자'] = pd.to_datetime(daily_request_count['접수일자'])
daily_request_count['접수월'] = daily_request_count['접수일자'].dt.month
daily_request_count['접수월주차'] = (
    daily_request_count['접수일자'].dt.strftime('%Y-%m')
    + '-'
    + (((daily_request_count['접수일자'].dt.day + daily_request_count['접수일자'].dt.to_period('M').dt.to_timestamp().dt.dayofweek - 1) // 7 + 1).astype(str))
    + '주차'
)

daily_request_count.head()

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

request_dt = pd.to_datetime(df_request['접수일시'], errors='coerce')

df_request_2025 = df_request[
    (request_dt >= '2025-01-01') &
    (request_dt < '2026-01-01')
].copy()

daily_request_count = (
    df_request_2025.assign(
        접수일자=pd.to_datetime(df_request_2025['접수일시'], errors='coerce').dt.date
    )
    .dropna(subset=['접수일자'])
    .groupby('접수일자')
    .size()
    .reset_index(name='접수건수')
)

daily_request_count['접수일자'] = pd.to_datetime(daily_request_count['접수일자'])

daily_request_count['7일이동평균'] = (
    daily_request_count['접수건수']
    .rolling(window=7, min_periods=1)
    .mean()
)

mean_count = daily_request_count['접수건수'].mean()
std_count = daily_request_count['접수건수'].std()

daily_request_count['z_score'] = (
    daily_request_count['접수건수'] - mean_count
) / std_count

outlier_dates = daily_request_count[
    daily_request_count['z_score'].abs() >= 3
]

plt.figure(figsize=(16, 6))

plt.plot(
    daily_request_count['접수일자'],
    daily_request_count['접수건수'],
    color='#4c78a8',
    linewidth=1,
    alpha=0.55,
    label='일별 접수건수'
)

plt.plot(
    daily_request_count['접수일자'],
    daily_request_count['7일이동평균'],
    color='#f58518',
    linewidth=2.5,
    label='7일 이동평균'
)

plt.scatter(
    outlier_dates['접수일자'],
    outlier_dates['접수건수'],
    color='red',
    s=45,
    label='이상치 후보'
)

plt.axhline(
    mean_count,
    color='gray',
    linestyle='--',
    linewidth=1,
    label=f'평균 {mean_count:,.0f}건'
)

plt.title('2025년 일자별 접수건수 추세')
plt.xlabel('접수일자')
plt.ylabel('접수건수')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'일평균 접수건수: {mean_count:,.2f}')
print(f'표준편차: {std_count:,.2f}')
print(f'이상치 후보 날짜 수: {len(outlier_dates):,}')

### 지역별 접수 수요

`출발구`, `목적구`, `출발구-목적구 조합` 기준으로 접수건수를 집계해 지역별 요청 집중도를 확인한다.



In [ ]:
# REGION_REQUEST_DEMAND_ANALYSIS_CELL
# 지역별 접수 수요 분석: 출발구, 목적구, 출발구 X 목적구 조합
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

if 'df_request' not in globals():
    df_request = pd.read_csv(
        'data/서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv'
    )

서울_25개구 = [
    '강남구', '강동구', '강북구', '강서구', '관악구',
    '광진구', '구로구', '금천구', '노원구', '도봉구',
    '동대문구', '동작구', '마포구', '서대문구', '서초구',
    '성동구', '성북구', '송파구', '양천구', '영등포구',
    '용산구', '은평구', '종로구', '중구', '중랑구',
]

출발구별_접수건수 = (
    df_request
    .dropna(subset=['출발구'])
    .groupby('출발구')
    .size()
    .reset_index(name='접수건수')
    .sort_values('접수건수', ascending=False)
    .reset_index(drop=True)
)

목적구별_접수건수 = (
    df_request
    .dropna(subset=['목적구'])
    .groupby('목적구')
    .size()
    .reset_index(name='접수건수')
    .sort_values('접수건수', ascending=False)
    .reset_index(drop=True)
)

출발구_top25 = 출발구별_접수건수.head(25).sort_values('접수건수')
목적구_top25 = 목적구별_접수건수.head(25).sort_values('접수건수')

fig, (ax_start, ax_dest) = plt.subplots(1, 2, figsize=(18, 10))

ax_start.barh(출발구_top25['출발구'], 출발구_top25['접수건수'], color='#4c78a8')
ax_start.set_title('출발구별 접수건수 TOP 25')
ax_start.set_xlabel('접수건수')
ax_start.set_ylabel('출발구')
ax_start.grid(axis='x', alpha=0.3)
for index, value in enumerate(출발구_top25['접수건수']):
    ax_start.text(value, index, f' {value:,.0f}', va='center', fontsize=8)

ax_dest.barh(목적구_top25['목적구'], 목적구_top25['접수건수'], color='#f58518')
ax_dest.set_title('목적구별 접수건수 TOP 25')
ax_dest.set_xlabel('접수건수')
ax_dest.set_ylabel('목적구')
ax_dest.grid(axis='x', alpha=0.3)
for index, value in enumerate(목적구_top25['접수건수']):
    ax_dest.text(value, index, f' {value:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

출발구_목적구_접수건수 = (
    df_request
    .dropna(subset=['출발구', '목적구'])
    .groupby(['출발구', '목적구'])
    .size()
    .reset_index(name='접수건수')
    .sort_values('접수건수', ascending=False)
    .reset_index(drop=True)
)

출발구_목적구_피벗 = (
    출발구_목적구_접수건수
    .pivot_table(
        index='출발구',
        columns='목적구',
        values='접수건수',
        aggfunc='sum',
        fill_value=0
    )
    .reindex(index=서울_25개구, columns=서울_25개구, fill_value=0)
)

plt.figure(figsize=(16, 13))
sns.heatmap(
    출발구_목적구_피벗,
    cmap='YlOrRd',
    linewidths=0.3,
    cbar_kws={'label': '접수건수'}
)
plt.title('출발구 X 목적구 접수건수 히트맵')
plt.xlabel('목적구')
plt.ylabel('출발구')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print('출발구별 접수건수 TOP 10')
display(출발구별_접수건수.head(10))

print('목적구별 접수건수 TOP 10')
display(목적구별_접수건수.head(10))

print('출발구 X 목적구 접수건수 TOP 20')
display(출발구_목적구_접수건수.head(20))

